# Fashion-MNIST DDLGN Depth vs Width Sweep

This notebook trains Differentiable Logic Gate Networks on Fashion-MNIST for the architecture grid shown in the following table: depths `1..6` and widths `2000, 4000, 6000, 8000` neurons per logic layer.

For every `(depth, width)` pair, the notebook creates a separate model directory under `trained_models_FashionMNIST_Depth_vs_Width`, saves the trained PyTorch model, exports the discretized logic-gate network as `best_lgn_gates.csv`, writes `best_lgn_metadata.json`, exports `test_binarized.csv`, and stores logs matching the existing `trained_models/<run_id>` folders.

The final section runs the Rust plaintext evaluator over all exported discretized models, prints an accuracy table by depth and width, and writes CSV/JSON/Markdown reports.

## Environment to activate

Use the same Windows CUDA environment described by `../../docs/DiffLogic_Installation_Guide.pdf` before starting Jupyter:

```powershell
conda activate difflogic2
cd <path-to-EI-DDLGN>
jupyter lab
```

Expected Python side:

- Python `3.10`
- PyTorch `2.5.1` with CUDA `12.1`
- CUDA Toolkit `12.1`
- `difflogic` built from the patched source described in the installation guide
- `difflogic_cuda` importable from the active environment

Expected Rust side:

- `cargo` available on `PATH`
- the repository's `MNIST_DDLGNs/lgn_eval/Cargo.toml` present

The next cell checks these assumptions before the long training sweep begins.

In [ ]:
# Preflight: project paths, CUDA DLLs, Python packages, and Rust tooling.
import os
import sys
import platform
import shutil
import subprocess
from pathlib import Path

REQUIRE_CUDA = True

def find_project_dir():
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "crates" / "ei-ddlgn-eval" / "Cargo.toml").exists():
            return candidate.resolve()
    raise RuntimeError("Could not locate MNIST_DDLGNs. Start Jupyter from the MNIST_DDLGNs folder or the repo root.")

PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)
print("Working directory:", PROJECT_DIR)
print("Python executable:", sys.executable)
print("Platform:", platform.platform())

import torch

if os.name == "nt":
    torch_lib_dir = Path(torch.__file__).resolve().parent / "lib"
    cuda_bin_dir = Path(r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin")
    for dll_dir in [torch_lib_dir, cuda_bin_dir]:
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))
            print("Added DLL directory:", dll_dir)
        else:
            print("DLL directory not found:", dll_dir)

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
elif REQUIRE_CUDA:
    raise RuntimeError("CUDA is not available. Activate the difflogic2 CUDA environment before running this notebook.")

import torchvision
import pandas as pd
import numpy as np
from tqdm import tqdm

import difflogic
from difflogic import LogicLayer, GroupSum, PackBitsTensor
from difflogic.compiled_model import ALL_OPERATIONS
print("difflogic:", difflogic.__file__)

if REQUIRE_CUDA:
    import difflogic_cuda
    print("difflogic_cuda loaded:", difflogic_cuda)

cargo_path = shutil.which("cargo")
if cargo_path is None:
    raise RuntimeError("cargo was not found on PATH. Install Rust with rustup and restart the terminal/Jupyter server.")
cargo_version = subprocess.run(["cargo", "--version"], capture_output=True, text=True, check=True).stdout.strip()
print("cargo:", cargo_version)

RUST_MANIFEST = PROJECT_DIR / "crates" / "ei-ddlgn-eval" / "Cargo.toml"
if not RUST_MANIFEST.exists():
    raise RuntimeError(f"Missing Rust manifest: {RUST_MANIFEST}")
print("Rust manifest:", RUST_MANIFEST)

## Sweep configuration

The grid matches the attached depth/width table:

- `depth`: number of `LogicLayer` layers, from `1` to `6`
- `width`: neurons per `LogicLayer`, one of `2000`, `4000`, `6000`, `8000`

The default training hyperparameters are inherited from `DDLGNs__MNIST.ipynb`: batch size `100`, learning rate `0.01`, early stopping patience `10`, seed `952`, and Fashion-MNIST `threshold_05` binarization. The `tau` values below follow the scale already used in the original notebook for the existing `small`, `medium`, and `large` runs. If you need exact paper hyperparameters for every cell in the table, edit `TAU_BY_WIDTH` before running the sweep.

In [ ]:
import datetime
import json
import random
import time
import pickle
import csv
import gc
from copy import deepcopy

import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import random_split

WIDTHS = [2000, 4000, 6000, 8000]
DEPTHS = [1, 2, 3, 4, 5, 6]
CLASS_COUNT = 10

TAU_BY_WIDTH = {
    2000: 3.0,
    4000: 3.0,
    6000: 5.0,
    8000: 10.0,
}

BASE_ARGS = {
    "dataset": "fashion_mnist",
    "id": 10,
    "batch_size": 100,
    "learning_rate": 0.01,
    "epochs": 200,
    "patience": 10,
    "img_size": 28,
    "seed": 952,
    "valid_set_size": 0.1,
    "pixel_mode": "threshold_05",  # "threshold_05" or "multi_threshold"
    "num_niveis": 3,
}

SWEEP_ROOT = PROJECT_DIR / "outputs" / "training" / "fashion-mnist"
SWEEP_RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
SWEEP_OUTPUT_DIR = SWEEP_ROOT / SWEEP_RUN_ID
SWEEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_SWEEP = True
SKIP_EXISTING = True

print("Sweep output directory:", SWEEP_OUTPUT_DIR)
print("Architectures:", [(depth, width) for depth in DEPTHS for width in WIDTHS])

## Dataset

This cell uses the Fashion-MNIST download folder and the same binarization logic as the original MNIST notebook. For `threshold_05`, every image becomes one binary `28 x 28` channel, so `in_dim = 784`. For `multi_threshold`, every image becomes `num_niveis` binary channels, so `in_dim = num_niveis * 784`.

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_mnist_transform(mode, num_niveis=3):
    if mode == "multi_threshold":
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(
                lambda x: torch.cat(
                    [(x > ((i + 1) / (float(num_niveis) + 1))).float() for i in range(num_niveis)],
                    dim=0,
                )
            ),
        ])
    if mode == "threshold_05":
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(lambda x: (x > 0.5).float()),
        ])
    raise ValueError(f"Unknown pixel_mode: {mode}")

set_seed(BASE_ARGS["seed"])
mnist_tf = make_mnist_transform(BASE_ARGS["pixel_mode"], BASE_ARGS["num_niveis"])

full_train = FashionMNIST(root="./data-fashion-mnist", train=True, download=True, transform=mnist_tf)
test_images = FashionMNIST(root="./data-fashion-mnist", train=False, download=True, transform=mnist_tf)

train_size = int((1 - BASE_ARGS["valid_set_size"]) * len(full_train))
val_size = len(full_train) - train_size
train_images, val_images = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(BASE_ARGS["seed"]),
)

train_loader = torch.utils.data.DataLoader(
    train_images,
    batch_size=BASE_ARGS["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=0,
)
val_loader = torch.utils.data.DataLoader(
    val_images,
    batch_size=BASE_ARGS["batch_size"],
    shuffle=False,
    drop_last=False,
    num_workers=0,
)
test_loader = torch.utils.data.DataLoader(
    test_images,
    batch_size=BASE_ARGS["batch_size"],
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

x0, y0 = next(iter(train_loader))
print("Train images:", len(train_images))
print("Val images:", len(val_images))
print("Test images:", len(test_images))
print("Batch shape:", x0.shape, y0.shape)

## Preview Fashion-MNIST images

Run this cell before training to inspect one sample from each Fashion-MNIST class. The preview uses the raw grayscale images, while training uses the binarized transform configured above.


In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw

FASHION_MNIST_CLASSES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

preview_dataset = FashionMNIST(
    root="./data-fashion-mnist",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

examples = {}
for image, label in preview_dataset:
    label = int(label)
    if label not in examples:
        examples[label] = image.squeeze(0)
    if len(examples) == len(FASHION_MNIST_CLASSES):
        break

cell_w, cell_h = 130, 110
img_size = 72
canvas = Image.new("RGB", (5 * cell_w, 2 * cell_h), "white")
draw = ImageDraw.Draw(canvas)

for label in range(len(FASHION_MNIST_CLASSES)):
    row, col = divmod(label, 5)
    x0, y0 = col * cell_w, row * cell_h
    arr = (examples[label].numpy() * 255).astype("uint8")
    img = Image.fromarray(arr, mode="L").resize((img_size, img_size), Image.Resampling.NEAREST).convert("RGB")
    canvas.paste(img, (x0 + (cell_w - img_size) // 2, y0 + 8))
    draw.text((x0 + 8, y0 + 84), f"{label}: {FASHION_MNIST_CLASSES[label]}", fill="black")

preview_path = PROJECT_DIR / "fashion_mnist_preview.png"
canvas.save(preview_path)
display(canvas)
print("Displayed one Fashion-MNIST sample image per class.")
print("Saved preview to:", preview_path)


## Training and export helpers

The export format is:

- `log_args.txt`
- `log_train.txt`
- `log_val.txt`
- `log_seconds.txt`
- `Model_depth-XX_width-YYYYY_epoch-Z`
- `best_lgn_gates.csv`
- `best_lgn_metadata.json`
- `test_binarized.csv`
- `results_boolean_depth-XX_width-YYYYY.txt`
- `training_summary.json`

`best_lgn_gates.csv` plus `best_lgn_metadata.json` are the saved discretized model representation. The Rust evaluator reconstructs the boolean network from those files.

In [ ]:
def make_args(depth, width):
    args = dict(BASE_ARGS)
    args.update({
        "num_layers": int(depth),
        "num_neurons": int(width),
        "tau": float(TAU_BY_WIDTH[int(width)]),
        "sweep_depth": int(depth),
        "sweep_width": int(width),
    })
    return args

def compute_in_dim(args):
    if args["pixel_mode"] == "multi_threshold":
        return int(args["num_niveis"] * args["img_size"] * args["img_size"])
    if args["pixel_mode"] == "threshold_05":
        return int(args["img_size"] * args["img_size"])
    raise ValueError(f"Unknown pixel_mode: {args['pixel_mode']}")

def build_lgn(args, device):
    in_dim = compute_in_dim(args)
    width = int(args["num_neurons"])
    depth = int(args["num_layers"])
    llkw = dict(grad_factor=1.0, connections="unique")

    logic_layers = [torch.nn.Flatten()]
    logic_layers.append(LogicLayer(in_dim=in_dim, out_dim=width, device=device, **llkw))
    for _ in range(depth - 1):
        logic_layers.append(LogicLayer(in_dim=width, out_dim=width, device=device, **llkw))

    model = torch.nn.Sequential(*logic_layers, GroupSum(CLASS_COUNT, args["tau"]))
    model = model.to(device)
    model.implementation = "cuda" if device == "cuda" else "python"

    total_num_neurons = sum(layer.num_neurons for layer in logic_layers[1:])
    total_num_weights = sum(layer.num_weights for layer in logic_layers[1:])
    return model, in_dim, total_num_neurons, total_num_weights

def train_step(model, x, y, loss_fn, optimizer):
    out = model(x)
    loss = loss_fn(out, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.item())

def _get_model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")

def eval_model(model, loader, mode=False, device=None):
    device = torch.device(device) if device is not None else _get_model_device(model)
    orig_mode = model.training
    with torch.no_grad():
        model.train(mode=mode)
        batch_accs = []
        for x, y in loader:
            x = x.to(torch.float32).to(device).round()
            y = y.to(device)
            batch_accs.append((model(x).argmax(-1) == y).to(torch.float32).mean().item())
        model.train(mode=orig_mode)
    return float(np.mean(batch_accs))

def export_lgn_gates_csv_and_json(model, csv_path, json_path, args, class_count, extra_metadata=None):
    rows = []
    for layer_idx, layer in enumerate(model):
        if isinstance(layer, LogicLayer):
            gate_ids = layer.weights.argmax(1).detach().cpu().tolist()
            a_idx = layer.indices[0].detach().cpu().tolist()
            b_idx = layer.indices[1].detach().cpu().tolist()
            for neuron_idx, (a, b, gate_id) in enumerate(zip(a_idx, b_idx, gate_ids)):
                rows.append({
                    "layer": int(layer_idx),
                    "neuron": int(neuron_idx),
                    "input_a": int(a),
                    "input_b": int(b),
                    "gate_id": int(gate_id),
                    "gate_name": str(ALL_OPERATIONS[gate_id]),
                })

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["layer", "neuron", "input_a", "input_b", "gate_id", "gate_name"],
        )
        writer.writeheader()
        writer.writerows(rows)

    pixel_mode = args.get("pixel_mode", "threshold_05")
    num_niveis = int(args.get("num_niveis", 1))
    img_size = int(args["img_size"])
    if pixel_mode == "multi_threshold":
        in_dim = num_niveis * img_size * img_size
        binarization = {
            "type": "levels_thresholds",
            "num_niveis": num_niveis,
            "thresholds": [(i + 1) / float(num_niveis + 1) for i in range(num_niveis)],
            "comparison": ">",
        }
    elif pixel_mode == "threshold_05":
        in_dim = img_size * img_size
        binarization = {"type": "single_threshold", "threshold": 0.5, "comparison": ">"}
    else:
        raise ValueError(f"Unknown pixel_mode: {pixel_mode}")

    logic_layer_sizes = [int(layer.out_dim) for layer in model if isinstance(layer, LogicLayer)]
    meta = {
        "dataset": args.get("dataset", "fashion_mnist"),
        "class_count": int(class_count),
        "img_size": img_size,
        "pixel_mode": pixel_mode,
        "num_niveis": num_niveis,
        "in_dim": int(in_dim),
        "num_layers": int(args["num_layers"]),
        "num_neurons": int(args["num_neurons"]),
        "tau": float(args["tau"]),
        "logic_layer_sizes": logic_layer_sizes,
        "gate_operations": list(ALL_OPERATIONS),
        "group_sum": {"type": "GroupSum", "class_count": int(class_count), "tau": float(args["tau"])},
        "preprocess": {
            "to_tensor": True,
            "binarization": binarization,
            "flatten_order": "row_major",
        },
        "csv_fields": ["layer", "neuron", "input_a", "input_b", "gate_id", "gate_name"],
        "discretized_model": {
            "gates_csv": "best_lgn_gates.csv",
            "metadata_json": "best_lgn_metadata.json",
            "rule": "gate_id is argmax over the learned 16 gate logits for each neuron",
        },
    }
    if extra_metadata:
        meta.update(extra_metadata)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

def export_binarized_test_csv(out_path):
    all_x = []
    all_y = []
    for x, y in test_loader:
        xb = x.round().bool().reshape(x.shape[0], -1).to(torch.int8).cpu()
        all_x.append(xb)
        all_y.append(y.cpu())
    x_all = torch.cat(all_x, dim=0)
    y_all = torch.cat(all_y, dim=0)
    df = pd.DataFrame(x_all.numpy())
    df.insert(0, "label", y_all.numpy())
    df.to_csv(out_path, index=False)
    return df.shape

def write_json(path, value):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2)

def train_and_export_model(depth, width):
    args = make_args(depth, width)
    model_dir = SWEEP_OUTPUT_DIR / f"depth_{depth:02d}_width_{width:05d}"
    model_dir.mkdir(parents=True, exist_ok=True)

    done_files = [
        model_dir / "best_lgn_gates.csv",
        model_dir / "best_lgn_metadata.json",
        model_dir / "test_binarized.csv",
        model_dir / "training_summary.json",
    ]
    if SKIP_EXISTING and all(p.exists() for p in done_files):
        print(f"Skipping existing export: depth={depth}, width={width}, dir={model_dir}")
        with open(model_dir / "training_summary.json", "r", encoding="utf-8") as f:
            return json.load(f)

    set_seed(args["seed"])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, in_dim, total_num_neurons, total_num_weights = build_lgn(args, device)
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args["learning_rate"])

    with open(model_dir / "log_args.txt", "w", encoding="utf-8") as f:
        f.write(json.dumps(args))

    print("=" * 80)
    print(f"Training depth={depth}, width={width}, tau={args['tau']}, device={device}")
    print("Model directory:", model_dir)
    print("in_dim:", in_dim, "total_num_neurons:", total_num_neurons, "total_num_weights:", total_num_weights)

    best_val = -1.0
    best_epoch = 0
    best_model = deepcopy(model)
    wait = 0
    train_list = []
    val_list = []
    loss_list = []
    seconds_list = []
    cumulative_seconds = 0.0

    for epoch in range(args["epochs"]):
        start_time = time.time()
        model.train()
        last_loss = None

        for x, y in tqdm(train_loader, desc=f"depth={depth} width={width} epoch={epoch}"):
            x = x.to(torch.float32).to(device)
            y = y.to(device)
            last_loss = train_step(model, x, y, loss_fn, optimizer)

        cumulative_seconds += time.time() - start_time
        seconds_list.append(round(cumulative_seconds, 0))

        val_acc = eval_model(model, val_loader, mode=False, device=device)
        train_acc = eval_model(model, train_loader, mode=False, device=device)
        val_list.append(val_acc)
        train_list.append(train_acc)
        loss_list.append(last_loss)

        print(
            f"epoch={epoch:03d} loss={last_loss:.6f} "
            f"train_acc={train_acc:.6f} val_acc={val_acc:.6f} best_val={best_val:.6f}"
        )

        if val_acc > best_val:
            best_val = val_acc
            best_epoch = epoch
            best_model = deepcopy(model)
            wait = 0
        else:
            wait += 1

        if wait > args["patience"]:
            print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
            break

    best_model = best_model.to(device)
    best_model.implementation = "cuda" if device == "cuda" else "python"
    test_acc = eval_model(best_model, test_loader, mode=False, device=device)

    model_name = f"Model_depth-{depth:02d}_width-{width:05d}_epoch-{best_epoch}"
    model_path = model_dir / model_name
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)

    write_json(model_dir / "log_seconds.txt", seconds_list)
    write_json(model_dir / "log_val.txt", val_list)
    write_json(model_dir / "log_train.txt", train_list)
    write_json(model_dir / "log_loss.txt", loss_list)

    export_lgn_gates_csv_and_json(
        best_model,
        model_dir / "best_lgn_gates.csv",
        model_dir / "best_lgn_metadata.json",
        args,
        CLASS_COUNT,
        extra_metadata={
            "sweep": {"depth": int(depth), "width": int(width), "run_id": SWEEP_RUN_ID},
            "best_epoch": int(best_epoch),
            "best_val_accuracy": float(best_val),
            "test_accuracy_python_discrete": float(test_acc),
            "model_file": model_name,
        },
    )
    test_csv_shape = export_binarized_test_csv(model_dir / "test_binarized.csv")

    result_text = (
        f"depth={depth}\n"
        f"width={width}\n"
        f"best_epoch={best_epoch}\n"
        f"best_val_accuracy={best_val:.16f}\n"
        f"test_accuracy_python_discrete={test_acc:.16f}\n"
        f"model_file={model_name}\n"
    )
    result_file = model_dir / f"results_boolean_depth-{depth:02d}_width-{width:05d}.txt"
    result_file.write_text(result_text, encoding="utf-8")

    summary = {
        "depth": int(depth),
        "width": int(width),
        "tau": float(args["tau"]),
        "best_epoch": int(best_epoch),
        "best_val_accuracy": float(best_val),
        "test_accuracy_python_discrete": float(test_acc),
        "training_seconds": float(cumulative_seconds),
        "model_dir": str(model_dir),
        "model_file": model_name,
        "test_csv_shape": list(test_csv_shape),
        "total_num_neurons": int(total_num_neurons),
        "total_num_weights": int(total_num_weights),
    }
    write_json(model_dir / "training_summary.json", summary)
    print(f"Saved model and discretized export to {model_dir}")
    print(f"Python discrete test accuracy: {test_acc:.6f}")

    del model, best_model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

## Run the depth vs width training sweep

This is the long-running cell. It trains `6 * 4 = 24` models. With the default `epochs=200` and early stopping, this can take a long time, especially for depth `6` and width `8000`.

If you want a smoke test before the full sweep, temporarily set `DEPTHS = [1]`, `WIDTHS = [2000]`, and `BASE_ARGS["epochs"]` to a small value in the configuration cell, then restart the kernel and run the notebook again.

In [ ]:
sweep_summaries = []

if RUN_SWEEP:
    for depth in DEPTHS:
        for width in WIDTHS:
            summary = train_and_export_model(depth, width)
            sweep_summaries.append(summary)
            pd.DataFrame(sweep_summaries).to_csv(SWEEP_OUTPUT_DIR / "training_summary_so_far.csv", index=False)
            write_json(SWEEP_OUTPUT_DIR / "training_summary_so_far.json", sweep_summaries)
else:
    print("RUN_SWEEP is False; training was skipped.")

if sweep_summaries:
    summary_df = pd.DataFrame(sweep_summaries).sort_values(["depth", "width"])
    summary_df.to_csv(SWEEP_OUTPUT_DIR / "training_summary.csv", index=False)
    display(summary_df[["depth", "width", "best_epoch", "best_val_accuracy", "test_accuracy_python_discrete", "training_seconds", "model_dir"]])

## Validate exported artifacts

Run this after training to verify that every model folder contains the same artifact categories as the existing trained model folders.

In [ ]:
REQUIRED_ARTIFACTS = [
    "log_args.txt",
    "log_val.txt",
    "log_train.txt",
    "log_seconds.txt",
    "best_lgn_gates.csv",
    "best_lgn_metadata.json",
    "test_binarized.csv",
    "training_summary.json",
]

def find_exported_model_dirs(root):
    root = Path(root)
    return sorted({p.parent for p in root.rglob("best_lgn_metadata.json")})

exported_dirs = find_exported_model_dirs(SWEEP_OUTPUT_DIR)
validation_rows = []
for model_dir in exported_dirs:
    with open(model_dir / "best_lgn_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    missing = [name for name in REQUIRED_ARTIFACTS if not (model_dir / name).exists()]
    model_files = sorted(p.name for p in model_dir.glob("Model_depth-*_width-*_epoch-*"))
    validation_rows.append({
        "depth": meta["num_layers"],
        "width": meta["num_neurons"],
        "model_dir": str(model_dir),
        "model_files": ", ".join(model_files),
        "missing": ", ".join(missing),
    })

validation_df = pd.DataFrame(validation_rows).sort_values(["depth", "width"])
display(validation_df)

if validation_df.empty:
    raise RuntimeError("No exported model folders found. Run the training sweep first.")
if validation_df["missing"].str.len().gt(0).any():
    raise RuntimeError("Some model folders are missing required artifacts. See the validation table above.")
print("All exported model folders contain the required artifacts.")

## Rust plaintext evaluation for all discretized models

This cell runs the repository Rust evaluator on each exported `best_lgn_gates.csv` + `best_lgn_metadata.json` + `test_binarized.csv` folder. It evaluates the saved discretized models, prints a depth-by-width accuracy table, and saves the report files into the sweep output directory.

Outputs:

- `rust_accuracy_report.csv`
- `rust_accuracy_report.json`
- `rust_accuracy_report.md`
- `rust_accuracy_pivot.csv`
- one `rust_eval_stdout.txt` per model folder

In [ ]:
import re

def latest_sweep_output_dir():
    candidates = [p for p in SWEEP_ROOT.iterdir() if p.is_dir()]
    if not candidates:
        raise RuntimeError("No sweep output directories found.")
    return max(candidates, key=lambda p: p.stat().st_mtime)

if "SWEEP_OUTPUT_DIR" in globals():
    EVAL_ROOT = Path(SWEEP_OUTPUT_DIR)
else:
    EVAL_ROOT = latest_sweep_output_dir()
print("Evaluating exported models under:", EVAL_ROOT)

def clean_cargo_env():
    """Avoid Conda MinGW compiler overrides when building the MSVC Rust target."""
    env = os.environ.copy()
    for key in [
        "CC", "CXX", "AR", "CFLAGS", "CXXFLAGS", "ARFLAGS",
        "HOST_CC", "HOST_CXX", "HOST_AR",
        "CC_x86_64_pc_windows_msvc", "CXX_x86_64_pc_windows_msvc", "AR_x86_64_pc_windows_msvc",
        "CC_x86_64-pc-windows-msvc", "CXX_x86_64-pc-windows-msvc", "AR_x86_64-pc-windows-msvc",
    ]:
        env.pop(key, None)
    return env

def lgn_eval_executable_path():
    exe_name = "lgn_eval.exe" if os.name == "nt" else "lgn_eval"
    return PROJECT_DIR / "crates" / "ei-ddlgn-eval" / "target" / "release" / exe_name

def build_rust_plaintext_evaluator():
    exe_path = lgn_eval_executable_path()
    if exe_path.exists():
        print("Using existing Rust evaluator:", exe_path)
        return exe_path
    cmd = ["cargo", "build", "--release", "--manifest-path", str(RUST_MANIFEST), "--bin", "lgn_eval"]
    print("Building Rust evaluator:", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_DIR, env=clean_cargo_env(), capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError("Cargo build failed. See the printed Cargo stdout/stderr above.")
    print(proc.stdout)
    print(proc.stderr)
    if not exe_path.exists():
        raise RuntimeError(f"Cargo build finished but evaluator was not found: {exe_path}")
    return exe_path

RUST_EVAL_EXE = build_rust_plaintext_evaluator()

def run_rust_plaintext_eval(model_dir):
    cmd = [str(RUST_EVAL_EXE), str(model_dir)]
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True)
    output = proc.stdout + "\n" + proc.stderr
    (Path(model_dir) / "rust_eval_stdout.txt").write_text(output, encoding="utf-8")
    if proc.returncode != 0:
        raise RuntimeError(f"Rust evaluator failed for {model_dir}. See rust_eval_stdout.txt in that folder.")
    match = re.search(r"Accuracy:\s*([0-9]*\.?[0-9]+)", output)
    if not match:
        raise RuntimeError(f"Could not parse Accuracy from Rust output for {model_dir}.")
    return float(match.group(1)), output

model_dirs = find_exported_model_dirs(EVAL_ROOT)
if not model_dirs:
    raise RuntimeError("No exported model directories found for Rust evaluation.")

rust_rows = []
for model_dir in model_dirs:
    with open(model_dir / "best_lgn_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    accuracy, _ = run_rust_plaintext_eval(model_dir)
    rust_rows.append({
        "depth": int(meta["num_layers"]),
        "width": int(meta["num_neurons"]),
        "accuracy": accuracy,
        "accuracy_percent": accuracy * 100.0,
        "best_epoch": int(meta.get("best_epoch", -1)),
        "best_val_accuracy": float(meta.get("best_val_accuracy", float("nan"))),
        "python_discrete_accuracy": float(meta.get("test_accuracy_python_discrete", float("nan"))),
        "model_dir": str(model_dir),
    })

rust_df = pd.DataFrame(rust_rows).sort_values(["depth", "width"])
pivot = rust_df.pivot(index="depth", columns="width", values="accuracy_percent").sort_index().sort_index(axis=1)

report_csv = EVAL_ROOT / "rust_accuracy_report.csv"
report_json = EVAL_ROOT / "rust_accuracy_report.json"
report_md = EVAL_ROOT / "rust_accuracy_report.md"
pivot_csv = EVAL_ROOT / "rust_accuracy_pivot.csv"

rust_df.to_csv(report_csv, index=False)
rust_df.to_json(report_json, orient="records", indent=2)
pivot.to_csv(pivot_csv)

def markdown_table_from_df(df, float_format=".4f"):
    headers = [str(column) for column in df.columns]
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for _, row in df.iterrows():
        values = []
        for value in row.tolist():
            if isinstance(value, float):
                values.append(format(value, float_format))
            else:
                values.append(str(value))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

report_lines = [
    "# Fashion-MNIST DDLGN Depth vs Width Rust Accuracy Report",
    "",
    f"Generated: {datetime.datetime.now().isoformat(timespec='seconds')}",
    f"Evaluation root: `{EVAL_ROOT}`",
    "",
    "## Per-model results",
    "",
    markdown_table_from_df(rust_df[["depth", "width", "accuracy", "accuracy_percent", "best_epoch", "model_dir"]]),
    "",
    "## Accuracy percent pivot",
    "",
    markdown_table_from_df(pivot.reset_index().round(4)),
    "",
]
report_md.write_text("\n".join(report_lines), encoding="utf-8")

print("Rust accuracy percent table:")
display(pivot.round(2))
print("Saved:", report_csv)
print("Saved:", report_json)
print("Saved:", report_md)
print("Saved:", pivot_csv)